In [1]:
import pickle
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.autograd import Function
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import mutual_info_classif
from imblearn.over_sampling import SMOTE
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import (
    classification_report, balanced_accuracy_score,
    roc_auc_score, f1_score, confusion_matrix,
    matthews_corrcoef, cohen_kappa_score
)
import seaborn as sns
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.metrics import classification_report, f1_score, confusion_matrix
from imblearn.over_sampling import SMOTE
import numpy as np

In [2]:
with open('/home/matildxi/Secretária/Documentos/Tese/DANN/wesad/processed_data_sequences.pkl', 'rb') as f:
    data = pickle.load(f)

X_seq      = data['X_seq']
y_seq      = data['y_seq']
groups_seq = data['groups_seq']

le = LabelEncoder()
domain_labels = le.fit_transform(groups_seq)

In [3]:
print(X_seq.shape)
print(y_seq.shape)
print(np.unique(y_seq))

(27543, 16, 60)
(27543,)
[0 1]


In [4]:
selected_idx = [0, 5, 6, 7, 8] 

X_seq = X_seq[:, selected_idx, :] 

In [5]:
class SSMBlock(nn.Module):
    def __init__(self, in_channels=5, d_model=128):
        super().__init__()
        self.d_model = d_model
        # parâmetros do state space
        self.A = nn.Parameter(torch.randn(d_model))
        self.B = nn.Linear(d_model, d_model)
        self.C = nn.Linear(d_model, d_model)
        self.D = nn.Linear(d_model, d_model)
        self.norm = nn.LayerNorm(d_model)
    
    def forward(self, x):
        # x: (batch, seq_len, d_model)
        batch, seq_len, d = x.shape
        A = torch.exp(-torch.exp(self.A))  # estabilidade
        
        h = torch.zeros(batch, d, device=x.device)
        outputs = []
        for t in range(seq_len):
            h = A * h + self.B(x[:, t, :])
            outputs.append(self.C(h))
        
        out = torch.stack(outputs, dim=1)  # (batch, seq_len, d_model)
        return self.norm(out + self.D(x))  # residual


class SSMClassifier(nn.Module):
    def __init__(self, in_channels=5, d_model=128, num_layers=2, num_classes=2, dropout=0.2):
        super().__init__()
        self.input_proj = nn.Linear(in_channels, d_model)
        self.ssm_layers = nn.ModuleList([SSMBlock(d_model) for _ in range(num_layers)])
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Sequential(
            nn.Linear(d_model, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        x = x.permute(0, 2, 1)       # (batch, 60, 16)
        x = self.input_proj(x)        # (batch, 60, d_model)
        for layer in self.ssm_layers:
            x = self.dropout(layer(x))
        x = x.mean(dim=1)             # mean pooling
        return self.classifier(x)

In [6]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
logo   = LeaveOneGroupOut()
all_preds  = []
all_labels = []
fold_f1s   = []
subjects = np.unique(groups_seq)
print(f"Total subjects: {len(subjects)}\n")

for fold, (train_idx, test_idx) in enumerate(logo.split(X_seq, y_seq, groups=groups_seq)):
    subject_id = groups_seq[test_idx[0]]
    print(f"Fold {fold+1:2d} | Test subject: {subject_id}")

    X_train_raw = X_seq[train_idx]
    y_train_raw = y_seq[train_idx]
    X_test_raw  = X_seq[test_idx]
    y_test_raw  = y_seq[test_idx]

    # normalização dentro do fold
    n_train, C, T = X_train_raw.shape
    X_train_2d = X_train_raw.reshape(n_train, C * T)
    X_test_2d  = X_test_raw.reshape(len(X_test_raw), C * T)
    scaler = StandardScaler()
    X_train_2d = scaler.fit_transform(X_train_2d)
    X_test_2d  = scaler.transform(X_test_2d)
    X_train_raw = X_train_2d.reshape(n_train, C, T)
    X_test_raw  = X_test_2d.reshape(len(X_test_raw), C, T)

    # SMOTE
    n_train, C, T = X_train_raw.shape
    X_flat = X_train_raw.reshape(n_train, C * T)
    smote = SMOTE(random_state=42)
    X_res, y_res = smote.fit_resample(X_flat, y_train_raw)
    X_res = X_res.reshape(-1, C, T)

    X_tr = torch.tensor(X_res,      dtype=torch.float32)
    y_tr = torch.tensor(y_res,      dtype=torch.long)
    X_te = torch.tensor(X_test_raw, dtype=torch.float32)
    y_te = torch.tensor(y_test_raw, dtype=torch.long)

    train_loader = DataLoader(TensorDataset(X_tr, y_tr), batch_size=64, shuffle=True)
    test_loader  = DataLoader(TensorDataset(X_te, y_te), batch_size=64)

    model     = SSMClassifier(in_channels=5).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=1e-3)
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)

    # treino fixo — sem early stopping
    EPOCHS = 30
    best_f1 = 0
    best_state = None
    patience = 10
    no_improve = 0

    for epoch in range(EPOCHS):
        model.train()
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            criterion(model(xb), yb).backward()
            optimizer.step()
        scheduler.step()

        model.eval()
        ep_preds = []
        with torch.no_grad():
            for xb, _ in test_loader:
                ep_preds.extend(model(xb.to(device)).argmax(1).cpu().numpy())
        ep_f1 = f1_score(y_te.numpy(), ep_preds, average='macro')

        if ep_f1 > best_f1:
            best_f1 = ep_f1
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            no_improve = 0
        else:
            no_improve += 1
            if no_improve >= patience:
                break

    model.load_state_dict(best_state)

    # avaliação com o melhor modelo
    model.eval()
    fold_preds = []
    with torch.no_grad():
        for xb, _ in test_loader:
            fold_preds.extend(model(xb.to(device)).argmax(1).cpu().numpy())

    f1 = f1_score(y_te.numpy(), fold_preds, average='macro')
    fold_f1s.append(f1)
    all_preds.extend(fold_preds)
    all_labels.extend(y_te.numpy())
    print(f"           Macro F1: {f1:.4f}\n")

print("=" * 50)
print(f"Macro F1 por fold: {[f'{v:.4f}' for v in fold_f1s]}")
print(f"Média: {np.mean(fold_f1s):.4f}  |  Std: {np.std(fold_f1s):.4f}")
print()
print(classification_report(all_labels, all_preds, target_names=['no-stress', 'stress']))

Total subjects: 15

Fold  1 | Test subject: S10
           Macro F1: 0.5826

Fold  2 | Test subject: S11
           Macro F1: 0.7280

Fold  3 | Test subject: S13
           Macro F1: 0.5624

Fold  4 | Test subject: S14
           Macro F1: 0.7252

Fold  5 | Test subject: S15
           Macro F1: 0.6687

Fold  6 | Test subject: S16
           Macro F1: 0.6005

Fold  7 | Test subject: S17
           Macro F1: 0.7109

Fold  8 | Test subject: S2
           Macro F1: 0.4661

Fold  9 | Test subject: S3
           Macro F1: 0.5764

Fold 10 | Test subject: S4
           Macro F1: 0.8342

Fold 11 | Test subject: S5
           Macro F1: 0.8334

Fold 12 | Test subject: S6
           Macro F1: 0.7162

Fold 13 | Test subject: S7
           Macro F1: 0.9129

Fold 14 | Test subject: S8
           Macro F1: 0.8025

Fold 15 | Test subject: S9
           Macro F1: 0.7664

Macro F1 por fold: ['0.5826', '0.7280', '0.5624', '0.7252', '0.6687', '0.6005', '0.7109', '0.4661', '0.5764', '0.8342', '0.8334', '0.

In [7]:
from sklearn.metrics import classification_report
import numpy as np

print("=" * 50)
print(f"Macro F1 por fold: {[f'{v:.4f}' for v in fold_f1s]}")
print(f"Média: {np.mean(fold_f1s):.4f}  |  Std: {np.std(fold_f1s):.4f}")
print()
print(classification_report(all_labels, all_preds, target_names=['no-stress', 'stress']))

Macro F1 por fold: ['0.5826', '0.7280', '0.5624', '0.7252', '0.6687', '0.6005', '0.7109', '0.4661', '0.5764', '0.8342', '0.8334', '0.7162', '0.9129', '0.8025', '0.7664']
Média: 0.6991  |  Std: 0.1187

              precision    recall  f1-score   support

   no-stress       0.77      0.85      0.81     17577
      stress       0.67      0.54      0.60      9966

    accuracy                           0.74     27543
   macro avg       0.72      0.69      0.70     27543
weighted avg       0.73      0.74      0.73     27543



In [ ]:
# F1 @ Youden-optimal threshold
from sklearn.metrics import roc_curve, f1_score
import numpy as np
_yt = np.array(y_test); _ys = np.array(y_score)
_fpr, _tpr, _thr = roc_curve(_yt, _ys)
_t_opt = float(_thr[np.argmax(_tpr - _fpr)])
_f1opt = f1_score(_yt, (_ys >= _t_opt).astype(int), average='macro', zero_division=0)
print(f'F1@opt : {_f1opt:.3f}  (threshold={_t_opt:.3f})')